---
## 🎁 가산점

### A. 데이터의 다양성
- NTP ICE 내 다양한 데이터셋 모두 활용 가능. (https://ice.ntp.niehs.nih.gov/DATASETDESCRIPTION)
### B. Feature(descriptor)의 다양성
- rdkit, VEGA, 등
### 💬 추가 설명 (자유 기술)

# 기말고사 Template 1 — Data Pipeline

**이름:** ______육건우________ &nbsp; **학번:** _______20251284_______ &nbsp;

---

## 📋 채점 기준 (총 50점)

| 항목 | 배점 | 채점 포인트 |
|---|---|---|
| **1. 데이터 분포 파악 및 전처리** | 15점 | 모델 개발 전, 중복 화합물 체크, smiles 코드 정리 등 모델 개발 전 확인해야 할 사항들을 확인. |
| **2. Descriptor 계산** | 15점 | 모델 개발에 사용할 descriptor의 다양성 |
| **3. 데이터 시각화 자료** | 15점 | 구조 분포, 라벨 비율 등 데이터 현황을 시각화한 자료 |
| **4. 코드 가독성 & 주석** | 5점 | 변수의 의미와 코드의 간결성을 평가. |

#### A. 데이터 소스의 다양성
- NTP ICE에서 구할 수 있는 다양한 데이터
- NTP ICE 외 추가 데이터 확보

## 📁 입력 / 출력 예시
- **입력**: `skin_irritation.xlsx` (NTP ICE) + (선택) 외부 데이터 (데이터가 여러개인 경우 xlsx파일의 zip으로 제출)
- **출력**: `final_dataset_descriptors.csv`  (Chemical_Name, SMILES, label, 2D descriptor [+ fingerprint 등])

In [3]:
import pandas as pd

file_path = 'cancer.xlsx' 
sheet_all = 'Data'

# pd.read_excel 함수를 사용해 특정 시트를 읽어온다.
df = pd.read_excel(file_path, sheet_name=sheet_all)

print(df.shape) #전체 데이터의 크기 행과 열의 개수 확인
print(df.columns) #전체 컬럼 이름 확인

(10351, 28)
Index(['Record_ID', 'Data_Type', 'Formulation_ID', 'Formulation_Name',
       'Chemical_Name', 'CASRN', 'DTXSID', 'Percent_Active_Ingredient',
       'Mixture', 'Species', 'Sex', 'Strain', 'Route', 'Level_of_Evidence',
       'Tissue', 'Location', 'Lesion', 'Assay', 'Endpoint', 'Response',
       'Response_Unit', 'Reference', 'URL', 'SMILES', 'Preferred_Name',
       'Synonyms', 'URL_CompTox', 'URL_CEBS'],
      dtype='object')


In [4]:
# 전체 데이터 중 'NTP Carcinogenicity' 관련 데이터만 골라내기
is_ntp = df["Endpoint"] == "Level of evidence of carcinogenic activity"
ntp_df = df[is_ntp].copy()

# 그중에서 생체 내(In Vivo) 실험 데이터만 남기기
ntp_in_vivo_df = ntp_df[ntp_df["Data_Type"] == "In Vivo"].copy()

# 분자 구조(SMILES)가 비어있는 행은 제외하기
ntp_in_vivo_df = ntp_in_vivo_df[ntp_in_vivo_df["SMILES"].notna()].copy()

# 현재까지 정제된 데이터의 크기 확인
ntp_in_vivo_df.shape

(2213, 28)

In [11]:
# 타겟 라벨 이진 분류 변환 매핑 정의
label_map = {"Clear evidence": 1, "Some evidence": 1, "No evidence": 0}

# 유효한 독성 평가 결과(라벨)를 가진 레코드만 필터링 및 변환
df_filtered = ntp_in_vivo_df[ntp_in_vivo_df["Response"].isin(label_map.keys())].copy()
df_filtered["label"] = df_filtered["Response"].map(label_map)

# 데이터 수가 적은 Hamster 제외 (Rat, Mouse만 사용)
df_filtered = df_filtered[df_filtered["Species"].isin(["Rat", "Mouse"])].copy()

df_filtered["Is_Rat"] = (df_filtered["Species"] == "Rat").astype(int)
df_filtered["Is_Male"] = (df_filtered["Sex"] == "Male").astype(int)

print(df_filtered.shape)
df_filtered

#Is_Rat=1 이면 Rat, 0이면 Mouse
#Is_Male=1 이면 수컷, 0이면 암컷

(1757, 31)


,Record_ID,Data_Type,Formulation_ID,Formulation_Name,Chemical_Name,CASRN,DTXSID,Percent_Active_Ingredient,Mixture,Species,...,Reference,URL,SMILES,Preferred_Name,Synonyms,URL_CompTox,URL_CEBS,label,Is_Rat,Is_Male
3,cancer_3383,In Vivo,NaN,Acronycine,Acronycine,7008-42-6,DTXSID0020026,NaN,Chemical,Rat,...,TR-049,https://ntp.niehs.nih.gov/publications/reports...,COC1=CC2=C(C=CC(C)(C)O2)C2=C1C(=O)C1=C(C=CC=C1...,Acronycine,"7008-42-6|Acronycine|3,12-Dihydro-6-methoxy-3,...",https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID0020026,1,1,1
9,cancer_3380,In Vivo,NaN,Acronycine,Acronycine,7008-42-6,DTXSID0020026,NaN,Chemical,Rat,...,TR-049,https://ntp.niehs.nih.gov/publications/reports...,COC1=CC2=C(C=CC(C)(C)O2)C2=C1C(=O)C1=C(C=CC=C1...,Acronycine,"7008-42-6|Acronycine|3,12-Dihydro-6-methoxy-3,...",https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID0020026,1,1,0
21,cancer_6845,In Vivo,NaN,Benzyl acetate,Benzyl acetate,140-11-4,DTXSID0020151,NaN,Chemical,Rat,...,TR-250,https://ntp.niehs.nih.gov/publications/reports...,CC(=O)OCC1=CC=CC=C1,Benzyl acetate,140-11-4|Benzyl acetate|(Acetoxymethyl)benzene...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID0020151,0,1,0
23,cancer_6844,In Vivo,NaN,Benzyl Acetate,Benzyl acetate,140-11-4,DTXSID0020151,NaN,Chemical,Rat,...,TR-431,https://ntp.niehs.nih.gov/publications/reports...,CC(=O)OCC1=CC=CC=C1,Benzyl acetate,140-11-4|Benzyl acetate|(Acetoxymethyl)benzene...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID0020151,0,1,0
24,cancer_6840,In Vivo,NaN,Benzyl acetate,Benzyl acetate,140-11-4,DTXSID0020151,NaN,Chemical,Mouse,...,TR-250,https://ntp.niehs.nih.gov/publications/reports...,CC(=O)OCC1=CC=CC=C1,Benzyl acetate,140-11-4|Benzyl acetate|(Acetoxymethyl)benzene...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID0020151,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10211,cancer_5373,In Vivo,NaN,alpha-Methylstyrene,alpha-Methylstyrene,98-83-9,DTXSID9025661,NaN,Chemical,Rat,...,TR-543,https://ntp.niehs.nih.gov/publications/reports...,CC(=C)C1=CC=CC=C1,alpha-Methylstyrene,98-83-9|alpha-Methylstyrene|(1-Methylethenyl)b...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID9025661,0,1,0
10214,cancer_5370,In Vivo,NaN,alpha-Methylstyrene,alpha-Methylstyrene,98-83-9,DTXSID9025661,NaN,Chemical,Rat,...,TR-543,https://ntp.niehs.nih.gov/publications/reports...,CC(=C)C1=CC=CC=C1,alpha-Methylstyrene,98-83-9|alpha-Methylstyrene|(1-Methylethenyl)b...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID9025661,1,1,1
10219,cancer_5369,In Vivo,NaN,alpha-Methylstyrene,alpha-Methylstyrene,98-83-9,DTXSID9025661,NaN,Chemical,Mouse,...,TR-543,https://ntp.niehs.nih.gov/publications/reports...,CC(=C)C1=CC=CC=C1,alpha-Methylstyrene,98-83-9|alpha-Methylstyrene|(1-Methylethenyl)b...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID9025661,1,0,0
10289,cancer_3034,In Vivo,NaN,Allyl Glycidyl Ether,Allyl glycidyl ether,106-92-3,DTXSID9039232,NaN,Chemical,Mouse,...,TR-376,https://ntp.niehs.nih.gov/publications/reports...,C=CCOCC1CO1,Allyl glycidyl ether,106-92-3|Allyl glycidyl ether|(.+-.)-Allyl gly...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID9039232,1,0,1


In [7]:
# [화합물 구조(SMILES) + 종 + 성별]이 같은 실험군 내에서 결과가 상충할 경우,
# 독성학적 안전성을 고려하여 단 한 번이라도 발암성(1)이 나왔다면 최종 1로 통합 (.max() 활용)
df_grouped = (
    df_filtered.groupby(["SMILES", "Is_Rat", "Is_Male"])
    .agg({"Chemical_Name": "first", "label": "max"})
    .reset_index()
)

print(f"필터링 및 중복 제거 후 최종 화합물 실험 데이터 개수: {df_grouped.shape[0]}개")

필터링 및 중복 제거 후 최종 화합물 실험 데이터 개수: 1682개
